In [ ]:
# Setup and imports
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use local development version of ras-commander
# (ensures RasTerrainMod is available even if not in pip-installed version)
dev_path = str(Path('.').resolve().parent)
if dev_path not in sys.path:
    sys.path.insert(0, dev_path)

# Force reimport from local source
import importlib
import ras_commander.terrain
importlib.reload(ras_commander.terrain)

from ras_commander.terrain import RasTerrainMod

print('RasTerrainMod imported successfully')

## Step 1: One-Time GDAL Bridge Setup

RasMapperLib.dll requires GDAL to be available relative to the Python executable. This creates a directory junction (symbolic link) from the Python installation to the HEC-RAS GDAL folder. Only needs to be run once per Python environment.

In [ ]:
# One-time setup - creates GDAL junction if it doesn't exist
bridge_ok = RasTerrainMod.setup_gdal_bridge()
print(f'GDAL bridge ready: {bridge_ok}')

## Step 2: Define Project Paths

Point to a HEC-RAS project that has terrain modifications. You need:
- The `.rasmap` file (references terrain layers and their modifications)
- A geometry `.g##.hdf` file (provides spatial reference)

In [ ]:
# === USER: Set your project paths here ===
project_folder = Path(r'G:\BayouConway_A14_03202026')
rasmap_path = project_folder / 'BayouConway_Update_250731.rasmap'
geom_hdf = project_folder / 'BayouConway_Update_250731.g17.hdf'

# Verify files exist
assert rasmap_path.exists(), f'rasmap not found: {rasmap_path}'
assert geom_hdf.exists(), f'geometry HDF not found: {geom_hdf}'
print(f'Project: {project_folder.name}')
print(f'Rasmap: {rasmap_path.name}')
print(f'Geometry: {geom_hdf.name}')

## Step 3: Get Terrain Extent

Query the terrain bounding box to understand the project domain.

In [ ]:
extent = RasTerrainMod.get_terrain_extent(rasmap_path, geom_hdf)
print(f"Success: {extent['success']}")
print(f"X range: {extent['min_x']:,.0f} to {extent['max_x']:,.0f}")
print(f"Y range: {extent['min_y']:,.0f} to {extent['max_y']:,.0f}")

## Step 4: Sample Terrain Profile with Modifications

Sample terrain along a polyline. The returned elevations include ALL terrain modifications (channels, levees, polygon overrides) that are defined in the terrain HDF referenced by the .rasmap file.

This is equivalent to cutting a cross-section through the terrain in RASMapper â€” but without opening the GUI.

In [ ]:
# Define a profile line across the project area
# (coordinates in project CRS - LA State Plane South, US feet)
x_profile = [3400000, 3402000, 3404000, 3406000, 3408000, 3410000]
y_profile = [612000, 612000, 612000, 612000, 612000, 612000]

# Sample terrain WITH modifications
profile = RasTerrainMod.get_terrain_profile(
    rasmap_path, geom_hdf,
    x_coords=x_profile,
    y_coords=y_profile
)

print(f'Profile points: {len(profile)}')
print(f'Station range: {profile["station"].min():.0f} to {profile["station"].max():.0f} ft')
print(f'Elevation range: {profile["elevation"].min():.2f} to {profile["elevation"].max():.2f} ft')
profile.head(10)

In [ ]:
# Plot the terrain profile
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(profile['station'], profile['elevation'], 'b-', linewidth=0.8)
ax.fill_between(profile['station'], profile['elevation'].min() - 0.5,
                profile['elevation'], alpha=0.2, color='sienna')
ax.set_xlabel('Station (ft)')
ax.set_ylabel('Elevation (ft)')
ax.set_title('Terrain Profile with Modifications Applied')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5: Elevation-Volume Curve for a Polygon

Compute the elevation-volume relationship within a polygon area. This is essential for:
- Detention pond sizing
- Storage volume calculations
- No-net-fill compliance

The volumes reflect the terrain WITH modifications â€” so if you've modeled a pond with terrain modifications, the volume curve includes the pond excavation.

In [ ]:
# Define an analysis polygon (1000 x 1000 ft area)
# Must close (first point == last point)
cx, cy = 3405000.0, 612000.0  # Center of analysis area
hw = 500.0  # Half-width

poly_x = [cx-hw, cx+hw, cx+hw, cx-hw, cx-hw]
poly_y = [cy-hw, cy-hw, cy+hw, cy+hw, cy-hw]

# Compute elevation-volume curve (in cubic feet)
ev_cuft = RasTerrainMod.get_terrain_volume_elevation(
    rasmap_path, geom_hdf,
    x_coords=poly_x, y_coords=poly_y,
    volume_factor=1.0  # cubic feet
)

# Also compute in acre-feet
ev_acft = RasTerrainMod.get_terrain_volume_elevation(
    rasmap_path, geom_hdf,
    x_coords=poly_x, y_coords=poly_y,
    volume_factor=43560.0  # acre-feet
)

print(f'Elevation-Volume table: {len(ev_cuft)} rows')
print(f'Elevation range: {ev_cuft["elevation"].min():.2f} to {ev_cuft["elevation"].max():.2f} ft')
print(f'Max volume: {ev_cuft["volume"].max():,.0f} cu ft = {ev_acft["volume"].max():.1f} ac-ft')

In [ ]:
# Plot elevation-volume curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ev_cuft['volume'], ev_cuft['elevation'], 'b-o', markersize=3)
ax1.set_xlabel('Volume (cubic feet)')
ax1.set_ylabel('Elevation (ft)')
ax1.set_title('Elevation-Volume Curve')
ax1.grid(True, alpha=0.3)
ax1.ticklabel_format(style='scientific', axis='x', scilimits=(0,0))

ax2.plot(ev_acft['volume'], ev_acft['elevation'], 'r-o', markersize=3)
ax2.set_xlabel('Volume (acre-feet)')
ax2.set_ylabel('Elevation (ft)')
ax2.set_title('Elevation-Volume Curve (acre-feet)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Elevation-Volume Table

Display the full elevation-volume table for engineering review.

In [ ]:
# Combine into a summary table
summary = ev_acft.copy()
summary.columns = ['Elevation (ft)', 'Volume (ac-ft)']
summary['Volume (cu ft)'] = ev_cuft['volume'].values[:len(summary)]
print(summary.to_string(index=False, float_format=lambda x: f'{x:,.2f}'))

## Key Takeaways

- **`RasTerrainMod`** provides headless access to HEC-RAS terrain sampling with modifications applied
- **No GUI required** â€” all operations use RasMapperLib.dll directly via pythonnet
- **Terrain modifications are automatically applied** â€” channels, levees, polygon overrides all included
- **One-time setup** â€” `setup_gdal_bridge()` creates a GDAL junction (only once per Python environment)
- **Cut/fill analysis** â€” compare profiles from existing vs proposed .rasmap files
- **No-net-fill compliance** â€” compare elevation-volume curves to verify excavation balances fill

## Adapting for Your Project

1. Update the project paths in Step 2
2. Adjust profile coordinates to match your project CRS and area of interest
3. Define analysis polygons around ponds, channels, or development areas
4. For no-net-fill: create two .rasmap files referencing existing and proposed terrain HDFs,
   then use `compare_terrain_volumes()` to compute the net fill at each elevation